# Import libraries

In [ ]:
# %pip install tensorly numpy matplotlib torch tqdm h5py scikit-image scikit-learn pandas
# %pip install --upgrade scikit-image reportlab

In [ ]:
# Standard library imports
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import numpy as np
import tensorly as tl
from pathlib import Path
from tqdm import tqdm
from typing import Optional, Tuple, Dict, List, Literal


# Local TBMD module imports
from TBMD.core.decomposition.hosvd import TuckerDecomposer
from TBMD.core.modal_processor.modes import (
    BatchModalProcessor, 
    ModalTensorStacker, 
    ModalProcessorConfig, 
    ProcessingStrategy,
)
from TBMD.core.sensor_placement.tensor_qr_factorization import (
    TensorTubeQRDecomposition,
    TensorQRConfig
)
# TBMD utilities imports
from TBMD.core.utils.misc import (
    build_wells_matrix
)
from TBMD.visualization import (
    visualize_tensor,
)
from TBMD.core.data.processors import (
    process_data, 
    calculate_global_minmax_params, 
    calculate_global_zscore_params, 
)
from TBMD.core.data.splitters import split_data_in_memory_ordered, split_data_in_memory

from TBMD.core.data.loaders import DataLoader
from TBMD.config import SEED, SET_BACKEND

In [ ]:
from TBMD.core.utils.misc import set_seed, set_torch_printoptions

# Use the centralized helper for reproducibility across libs
set_seed(SEED)
np.random.seed(SEED)

# Set TensorLy backend
tl.set_backend(SET_BACKEND)

# Safe printoptions across PyTorch versions
set_torch_printoptions(precision=4, sci_mode=False)

# Download data

In [ ]:
loader = DataLoader()

### Load Brugge data

(x,y, variants_geo, count_var, timestemp)

In [ ]:
import os

# Local dataset root -- set TBMD_DATA_DIR (see .env.example) to your local
# copy of the Brugge dataset before running this cell.
TBMD_DATA_DIR = os.environ.get("TBMD_DATA_DIR")
if not TBMD_DATA_DIR:
    raise RuntimeError(
        "Set the TBMD_DATA_DIR environment variable to your local Brugge "
        "dataset directory before running this notebook (see .env.example)."
    )

# Проверка правильности реализации

# Списки путей к данным и скважинам
data_paths = [
    f"{TBMD_DATA_DIR}/brugge/data_exp_5_1.h5",
    f"{TBMD_DATA_DIR}/brugge/data_exp_5_5.h5",
    f"{TBMD_DATA_DIR}/brugge/data_exp_5_10.h5",
    f"{TBMD_DATA_DIR}/brugge/data_exp_5_15.h5",
    f"{TBMD_DATA_DIR}/brugge/data_exp_5_20.h5",
    f"{TBMD_DATA_DIR}/brugge/data_exp_5_25.h5",
    f"{TBMD_DATA_DIR}/brugge/data_exp_5_30.h5"
]
wells_paths = [
    f"{TBMD_DATA_DIR}/brugge/all_wells_exp_5_1.json",
    f"{TBMD_DATA_DIR}/brugge/all_wells_exp_5_5.json",
    f"{TBMD_DATA_DIR}/brugge/all_wells_exp_5_10.json",
    f"{TBMD_DATA_DIR}/brugge/all_wells_exp_5_15.json",
    f"{TBMD_DATA_DIR}/brugge/all_wells_exp_5_20.json",
    f"{TBMD_DATA_DIR}/brugge/all_wells_exp_5_25.json",
    f"{TBMD_DATA_DIR}/brugge/all_wells_exp_5_30.json"
]

tensors_list = []
wells_list = []

# Проверяем, что количество путей совпадает
assert len(data_paths) == len(wells_paths), "Количество файлов данных и файлов скважин должно совпадать"

for d_path, w_path in zip(data_paths, wells_paths):
    # Загрузка тензоров
    tensor = loader.load_h5_tensors(d_path)
    tensors_list.append(tensor)

    # Загрузка скважин
    wells = loader.load_wells_from_json(w_path)
    # Проверка структуры wells: должен быть словарь с case_id -> список координат
    assert isinstance(wells, dict), f"wells должен быть dict, а не {type(wells)}"
    for case_id in wells:
        # Проверяем, что каждая точка - пара координат
        wells[case_id] = [[x, y] for x, y in wells[case_id]]
        for pair in wells[case_id]:
            assert isinstance(pair, list) and len(pair) == 2, f"Координаты должны быть списком из двух элементов, а не {pair}"
    wells_list.append(wells)


In [ ]:
# train_data, test_data = split_data_in_memory_ordered(tensors['all'], train_ratio=0.8)

# subject_name = list(tensors['all'].keys())[0]

# print(list(tensors['all'].keys()))
# print(tensors['all'][subject_name].shape)

In [ ]:
train_data_list = []
test_data_list = []

for tensor in tensors_list:
    train_data, test_data = split_data_in_memory_ordered(tensor['pressure'], train_ratio=0.8)
    train_data_list.append(train_data)
    test_data_list.append(test_data)
    for subject_name in tensor['pressure'].keys():
        print(subject_name)
        print(tensor['pressure'][subject_name].shape)

In [ ]:
# train_data_list = []
# test_data_list = []

# for tensor in tensors_list:
#     train_data, test_data = split_data_in_memory_ordered(tensor['soil'], train_ratio=0.8)
#     train_data_list.append(train_data)
#     test_data_list.append(test_data)
#     for subject_name in tensor['soil'].keys():
#         print(subject_name)
#         print(tensor['soil'][subject_name].shape)

### Load static csv data

In [ ]:
# # Load static tensor
# static_data = loader.load_data(Path(f"{TBMD_DATA_DIR}/HW static data"), "static", (286, 105, 100), tensor_type="pt")

# noisy_datasets = generate_noisy_datasets(
#     data=static_data,
#     noise_level=0.1,
#     num_noisy_datasets=5,
#     experiment_id="001"
# )

# train_data, test_data = split_data_in_memory_ordered(noisy_datasets, train_ratio=0.8)

# subject_name = list(noisy_datasets.keys())[0]

# print(list(noisy_datasets.keys()))
# print(noisy_datasets[subject_name].shape)

### Load dynamic csv data

In [ ]:
# # Load dynamic tensor
# dynamic_data = loader.load_data(Path(f"{TBMD_DATA_DIR}/HW dynamic data"), "dynamic", (286, 105, 25, 253), tensor_type="pt")

# train_data, test_data = split_data_in_memory_ordered(dynamic_data, train_ratio=0.8)

# subject_name = list(dynamic_data.keys())[0]

# print(list(dynamic_data.keys()))
# print(dynamic_data[subject_name].shape)

### Load images

In [ ]:
# # Load images tensor
# images_data, subject_list = loader.load_data(Path(f"{TBMD_DATA_DIR}/heriot_watt/dynamic_png_new"), "images", tensor_type="pt")

# train_data, test_data = split_data_in_memory_ordered(images_data, train_ratio=0.8)

# subject_name = subject_list[0]

# print(subject_list)
# print(images_data[subject_name].shape)

In [ ]:
# num_experiments = 2
# experiments_data = split_data_in_memory(images_data, num_experiments=num_experiments, train_ratio=0.8)

# if 1 not in experiments_data:
#     raise KeyError("Experiment ID 1 does not exist in 'experiments_data'.")

# train_data = experiments_data[1].get("train", {})
# test_data = experiments_data[1].get("test", {})

# Process data

In [ ]:
# Decide what counts as background (CT example)
BG = None        # Hounsfield Units for air

# Для всех train_data_list (по subject) сохранить результаты в список
minmax_params_list = []
zscore_params_list = []

for train_data in train_data_list:
    subj_min, subj_max = calculate_global_minmax_params(train_data, background_value=BG)
    subj_mean, subj_std = calculate_global_zscore_params(train_data, background_value=BG)
    minmax_params_list.append({'min': subj_min, 'max': subj_max})
    zscore_params_list.append({'mean': subj_mean, 'std': subj_std})

print("Minmax params list:", minmax_params_list)
print("Zscore params list:", zscore_params_list)

In [ ]:
resize_shape = None
convert_to_grayscale = False
normalization_method = "minmax"  # "zscore" or "minmax"

print("Processing train data:")
train_tensors_list = []
for i, train_data in enumerate(train_data_list):
    tensors = process_data(
        train_data,
        resize_shape=resize_shape,
        convert_to_grayscale=convert_to_grayscale,
        normalization_method=normalization_method,
        global_params=minmax_params_list[i],  # индивидуальные параметры для каждого subject
        background_value=BG
    )
    train_tensors_list.append(tensors)

print("\nProcessing test data:")
test_tensors_list = []
for i, test_data in enumerate(test_data_list):
    tensors = process_data(
        test_data,
        resize_shape=resize_shape,
        convert_to_grayscale=convert_to_grayscale,
        normalization_method=normalization_method,
        global_params=minmax_params_list[i],  # используем параметры train для соответствующего subject
        background_value=BG
    )
    test_tensors_list.append(tensors)

num_images_train = []
for tensors in train_tensors_list:
    num_images = {subject: tensor.shape[-1] for subject, tensor in tensors.items()}
    num_images_train.append(num_images)

num_images_test = []
for tensors in test_tensors_list:
    num_images = {subject: tensor.shape[-1] for subject, tensor in tensors.items()}
    num_images_test.append(num_images)

if num_images_train:
    min_train_images = min(
        min(subject_dict.values()) for subject_dict in num_images_train if subject_dict
)
    print(f"\nMinimum number of images in train: {min_train_images}")
else:
    print("\nNo data available for analysis in train.")

if num_images_test:
    min_test_images = min(
        min(subject_dict.values()) for subject_dict in num_images_test if subject_dict
)
    print(f"Minimum number of images in test: {min_test_images}")
else:
    print("No data available for analysis in test.")

# Visualization

In [ ]:
tensor = train_data_list[1][subject_name]
wells_swapped = {k: [[y, x] for x, y in v] for k, v in wells_list[0].items()}

visualize_tensor(tensor, subject_name, cmap="viridis", show_colorbar=True, save_path=None, wells=wells_swapped)  

In [ ]:
tensor = test_data_list[1][subject_name]
visualize_tensor(tensor, subject_name, cmap="viridis", show_colorbar=True, save_path=None, wells=wells_swapped)  

# Pipline

In [ ]:
# ============================================================
# 1. Visualization Utilities
# ============================================================

def visualize_combined_placement(
    P: torch.Tensor,
    wells: torch.Tensor,
    figsize: Optional[Tuple[int, int]] = None,
    titles: Tuple[str, str, str] = (
        "Sensors Only",
        "Wells Only",
        "Combined Placement"
    )
) -> None:
    """
    Visualize sensor and well positions on a grid.

    Displays three subplots:
    1) Sensors alone
    2) Wells alone
    3) Combined view (sensors + wells)

    Args:
        P (torch.Tensor): (H, W) binary tensor for sensor locations (1 = sensor).
        wells (torch.Tensor): (H, W) binary tensor for well locations (1 = well).
        figsize (tuple, optional): Figure size (width, height) in inches.
                                   Defaults to None (auto scaling).
        titles (tuple): Titles for the three subplots.
    """
    p_np = P.detach().cpu().numpy()
    wells_np = wells.detach().cpu().numpy()
    H, W = p_np.shape

    if figsize is None:
        figsize = (max(8, W / 5), max(4, H / 5))

    fig, axes = plt.subplots(1, 3, figsize=figsize, constrained_layout=True)
    background = np.zeros((H, W))

    # --- 1) Sensors only ---
    ax = axes[0]
    ax.set_facecolor("black")
    ax.imshow(background, cmap="gray", origin="upper")
    sensor_pos = np.argwhere(p_np == 1)
    if sensor_pos.size:
        ax.scatter(sensor_pos[:, 1], sensor_pos[:, 0],
                   s=50, c="red", marker="o", alpha=0.8, label="Sensors")
        ax.legend(loc="upper right")
    ax.set_title(titles[0], color="white")
    ax.axis("off")

    # --- 2) Wells only ---
    ax = axes[1]
    ax.set_facecolor("black")
    ax.imshow(background, cmap="gray", origin="upper")
    well_pos = np.argwhere(wells_np == 1)
    if well_pos.size:
        ax.scatter(well_pos[:, 1], well_pos[:, 0],
                   s=50, c="blue", marker="o", alpha=0.8, label="Wells")
        ax.legend(loc="upper right")
    ax.set_title(titles[1], color="white")
    ax.axis("off")

    # --- 3) Combined ---
    ax = axes[2]
    ax.set_facecolor("black")
    ax.imshow(background, cmap="gray", origin="upper")
    if sensor_pos.size:
        ax.scatter(sensor_pos[:, 1], sensor_pos[:, 0],
                   s=50, c="red", marker="o", alpha=0.8, label="Sensors")
    if well_pos.size:
        ax.scatter(well_pos[:, 1], well_pos[:, 0],
                   s=50, c="blue", marker="o", alpha=0.8, label="Wells")
    ax.legend(loc="upper right")
    ax.set_title(titles[2], color="white")
    ax.axis("off")

    plt.show()


def visualize_overlap_analysis(
    P: torch.Tensor,
    wells: torch.Tensor,
    analysis_result: Dict,
    figsize: Optional[Tuple[int, int]] = None
) -> None:
    """
    Visualize overlap analysis between sensors and wells using color-coded markers.

    Args:
        P (torch.Tensor): Binary tensor for sensor positions.
        wells (torch.Tensor): Binary tensor for well positions.
        analysis_result (dict): Output of `analyze_sensor_well_overlap`.
        figsize (tuple, optional): Figure size (width, height).
    """
    p_np = P.detach().cpu().numpy()
    wells_np = wells.detach().cpu().numpy()
    H, W = p_np.shape

    if figsize is None:
        figsize = (max(8, W / 5) / 3, max(4, H / 5))

    fig, ax = plt.subplots(1, 1, figsize=figsize, constrained_layout=True)
    background = np.zeros((H, W))
    ax.set_facecolor("black")
    ax.imshow(background, cmap="gray", origin="upper")

    sensor_positions = analysis_result['sensor_positions']
    well_positions = analysis_result['well_positions']
    overlapping_positions = analysis_result['overlapping_positions_list']
    wells_without_sensors = analysis_result['wells_without_sensors_indices']

    # --- Wells without sensors (blue) ---
    wells_no_sensors_positions = [well_positions[i] for i in wells_without_sensors]
    if wells_no_sensors_positions:
        wells_no_sensors_positions = np.array(wells_no_sensors_positions)
        ax.scatter(wells_no_sensors_positions[:, 1], wells_no_sensors_positions[:, 0],
                   s=100, c="blue", marker="o", alpha=0.8,
                   label=f"Wells without sensors ({len(wells_no_sensors_positions)})")

    # --- Sensors without wells (red) ---
    sensor_positions_array = np.array(sensor_positions)
    overlapping_positions_array = np.array(overlapping_positions) if overlapping_positions else np.array([])
    sensors_only = [s for s in sensor_positions_array if not any(np.array_equal(s, o) for o in overlapping_positions_array)]
    if sensors_only:
        sensors_only = np.array(sensors_only)
        ax.scatter(sensors_only[:, 1], sensors_only[:, 0],
                   s=100, c="red", marker="o", alpha=0.8,
                   label=f"Sensors without wells ({len(sensors_only)})")

    # --- Overlaps (green) ---
    if len(overlapping_positions_array) > 0:
        ax.scatter(overlapping_positions_array[:, 1], overlapping_positions_array[:, 0],
                   s=150, c="green", marker="*", alpha=1.0,
                   label=f"Overlaps ({len(overlapping_positions_array)})")

    ax.legend(loc="upper right")
    ax.set_title("Sensor-Well Overlap Analysis", color="white")
    ax.axis("off")
    plt.show()


# ============================================================
# 2. Overlap Analysis
# ============================================================

def analyze_sensor_well_overlap(
    P: torch.Tensor,
    wells: torch.Tensor,
    verbose: bool = True,
    proximity_radius: int = 2,
    proximity_metric: Literal['manhattan', 'euclidean', 'chebyshev'] = 'manhattan'
) -> Dict:
    """
    Analyze overlaps and nearby sensor proximity for each well.

    Args:
        P (torch.Tensor): Binary tensor for sensor locations (1 = sensor).
        wells (torch.Tensor): Binary tensor for well locations (1 = well).
        verbose (bool): If True, prints a detailed summary.
        proximity_radius (int): Distance threshold for "nearby" sensors.
        proximity_metric (str): Distance metric: 'manhattan', 'euclidean', or 'chebyshev'.

    Returns:
        dict: Metrics and overlap/proximity analysis.
    """
    p_np = P.detach().cpu().numpy()
    wells_np = wells.detach().cpu().numpy()

    sensor_positions = np.argwhere(p_np == 1)
    well_positions = np.argwhere(wells_np == 1)
    num_sensors = len(sensor_positions)
    num_wells = len(well_positions)

    overlapping_positions = []
    overlapping_sensor_indices = []
    overlapping_well_indices = []

    for i, sensor_pos in enumerate(sensor_positions):
        for j, well_pos in enumerate(well_positions):
            if np.array_equal(sensor_pos, well_pos):
                overlapping_positions.append(sensor_pos)
                overlapping_sensor_indices.append(i)
                overlapping_well_indices.append(j)

    # Remove duplicates
    unique_overlaps = []
    unique_sensor_idx = []
    unique_well_idx = []
    for pos, s_idx, w_idx in zip(overlapping_positions, overlapping_sensor_indices, overlapping_well_indices):
        if pos.tolist() not in [p.tolist() for p in unique_overlaps]:
            unique_overlaps.append(pos)
            unique_sensor_idx.append(s_idx)
            unique_well_idx.append(w_idx)

    # Wells without exact sensors
    wells_without_sensors = []
    wells_with_nearby_sensors = []
    nearby_offsets = []

    for j, well_pos in enumerate(well_positions):
        if any(np.array_equal(well_pos, s) for s in sensor_positions):
            continue  # already exact match
        wells_without_sensors.append(j)

        found_nearby = False
        min_distance = float('inf')
        for sensor_pos in sensor_positions:
            delta = well_pos - sensor_pos
            if proximity_metric == 'manhattan':
                dist = np.abs(delta).sum()
            elif proximity_metric == 'euclidean':
                dist = np.linalg.norm(delta)
            elif proximity_metric == 'chebyshev':
                dist = np.abs(delta).max()
            else:
                raise ValueError(f"Unsupported metric: {proximity_metric}")

            if dist <= proximity_radius:
                found_nearby = True
                min_distance = min(min_distance, dist)

        if found_nearby:
            wells_with_nearby_sensors.append(j)
            nearby_offsets.append(min_distance)

    num_overlapping = len(unique_overlaps)
    num_wells_without_sensors = len(wells_without_sensors)
    num_wells_with_nearby = len(wells_with_nearby_sensors)
    num_fully_uncovered = num_wells_without_sensors - num_wells_with_nearby
    overlap_percentage = (num_overlapping / num_wells * 100) if num_wells > 0 else 0

    result = {
        'total_sensors': num_sensors,
        'total_wells': num_wells,
        'overlapping_positions': num_overlapping,
        'wells_without_sensors': num_wells_without_sensors,
        'wells_with_nearby_sensors': num_wells_with_nearby,
        'fully_uncovered_wells': num_fully_uncovered,
        'overlap_percentage': overlap_percentage,
        'sensor_positions': sensor_positions,
        'well_positions': well_positions,
        'overlapping_positions_list': unique_overlaps,
        'wells_without_sensors_indices': wells_without_sensors,
        'wells_with_nearby_sensor_offsets': nearby_offsets,
        'proximity_radius': proximity_radius,
        'proximity_metric': proximity_metric
    }

    if verbose:
        print("=" * 60)
        print("SENSOR-WELL OVERLAP ANALYSIS WITH PROXIMITY CHECK")
        print("=" * 60)
        print(f"Total sensors:               {num_sensors}")
        print(f"Total wells:                 {num_wells}")
        print(f"Exact overlaps:              {num_overlapping}")
        print(f"Wells without exact sensors: {num_wells_without_sensors}")
        print(f"↳ Nearby within radius={proximity_radius} (metric='{proximity_metric}'): {num_wells_with_nearby}")
        print(f"↳ Fully uncovered wells:     {num_fully_uncovered}")
        print(f"Exact coverage:              {overlap_percentage:.2f}%")
        print("=" * 60)

    return result


def get_overlap_statistics(P: torch.Tensor, wells: torch.Tensor) -> Dict:
    """
    Returns concise overlap statistics.

    Args:
        P (torch.Tensor): Sensor tensor.
        wells (torch.Tensor): Well tensor.

    Returns:
        dict: Summary statistics.
    """
    result = analyze_sensor_well_overlap(P, wells, verbose=False)
    return {
        'total_sensors': result['total_sensors'],
        'total_wells': result['total_wells'],
        'overlapping_count': result['overlapping_positions'],
        'wells_without_sensors_count': result['wells_without_sensors'],
        'overlap_percentage': result['overlap_percentage']
    }


# ============================================================
# 3. Multiple Placement Comparison
# ============================================================

def compare_multiple_placements(
    P_list: List[torch.Tensor],
    wells_list: List[torch.Tensor],
    labels: Optional[List[str]] = None
) -> Dict:
    """
    Compare multiple sensor placements against wells.

    Args:
        P_list (list): List of sensor placement tensors.
        wells_list (list): List of well tensors (or a single tensor used for all).
        labels (list): Custom labels for each placement.

    Returns:
        dict: Comparison results.
    """
    if labels is None:
        labels = [f"Placement {i+1}" for i in range(len(P_list))]
    if len(wells_list) == 1:
        wells_list = wells_list * len(P_list)

    results = {}
    for P, wells, label in zip(P_list, wells_list, labels):
        stats = get_overlap_statistics(P, wells)
        results[label] = stats

    print("=" * 80)
    print("COMPARATIVE SENSOR PLACEMENT ANALYSIS")
    print("=" * 80)
    print(f"{'Label':<20} {'Sensors':<10} {'Wells':<10} "
          f"{'Overlaps':<12} {'No Sensors':<12} {'Coverage %':<10}")
    print("-" * 80)
    for label, stats in results.items():
        print(f"{label:<20} {stats['total_sensors']:<10} {stats['total_wells']:<10} "
              f"{stats['overlapping_count']:<12} {stats['wells_without_sensors_count']:<12} "
              f"{stats['overlap_percentage']:<10.2f}")
    print("=" * 80)
    return results

In [ ]:
def save_textual_report(
    subject_index: int,
    subject_name: str,
    analysis_result: dict,
    output_dir: str = "reports"
) -> None:
    """
    Save a plain-text summary of sensor-well analysis for a given subject.
    Also appends summary to global file 'summary_all.txt'.
    """
    subject_dir = os.path.join(output_dir, f"subject_{subject_index}")
    os.makedirs(subject_dir, exist_ok=True)

    # --- Prepare summary string ---
    summary = []
    summary.append(f"Subject Index     : {subject_index}")
    summary.append(f"Subject Name      : {subject_name}")
    summary.append(f"Total Sensors     : {analysis_result['total_sensors']}")
    summary.append(f"Total Wells       : {analysis_result['total_wells']}")
    summary.append(f"Exact Overlaps    : {analysis_result['overlapping_positions']}")
    summary.append(f"Wells w/o Sensors : {analysis_result['wells_without_sensors']}")
    
    if 'wells_with_nearby_sensors' in analysis_result:
        summary.append(f"Nearby (≤{analysis_result['proximity_radius']}, {analysis_result['proximity_metric']}): {analysis_result['wells_with_nearby_sensors']}")
        summary.append(f"Fully Uncovered   : {analysis_result['fully_uncovered_wells']}")
    
    summary.append(f"Coverage (%)      : {analysis_result['overlap_percentage']:.2f}")
    summary.append("-" * 50)

    # --- Save to per-subject txt file ---
    subject_txt_path = os.path.join(subject_dir, "summary.txt")
    with open(subject_txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(summary))

    # --- Append to global summary file ---
    global_summary_path = os.path.join(output_dir, "summary_all.txt")
    with open(global_summary_path, "a", encoding="utf-8") as f:
        f.write(f"{subject_index:<6} {subject_name:<15} "
                f"{analysis_result['total_sensors']:<8} {analysis_result['total_wells']:<8} "
                f"{analysis_result['overlapping_positions']:<8} {analysis_result['wells_without_sensors']:<8} "
        )

        if 'wells_with_nearby_sensors' in analysis_result:
            f.write(f"{analysis_result['wells_with_nearby_sensors']:<8} {analysis_result['fully_uncovered_wells']:<8} ")

        f.write(f"{analysis_result['overlap_percentage']:.2f}\n")

def append_to_pandas_report(
    subject_index: int,
    subject_name: str,
    analysis_result: dict,
    output_path: str = "reports/summary_all.csv"
) -> None:
    """
    Append a row of analysis to the global pandas DataFrame and save it to CSV.
    Creates the file if it doesn't exist.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Формируем строку результата
    row = {
        "Index": subject_index,
        "Subject Name": subject_name,
        "Total Sensors": analysis_result["total_sensors"],
        "Total Wells": analysis_result["total_wells"],
        "Exact Overlap": analysis_result["overlapping_positions"],
        "No Sensors": analysis_result["wells_without_sensors"],
        "Nearby Sensors": analysis_result.get("wells_with_nearby_sensors", None),
        "Fully Uncovered": analysis_result.get("fully_uncovered_wells", None),
        "Coverage (%)": round(analysis_result["overlap_percentage"], 2)
    }

    # Загружаем или создаём DataFrame
    if os.path.exists(output_path):
        df = pd.read_csv(output_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    # Сохраняем обратно
    df.to_csv(output_path, index=False, float_format="%.2f")


In [ ]:
# Initialize global summary file
os.makedirs("reports", exist_ok=True)

global_summary_path = "reports/summary_all.txt"
with open(global_summary_path, "w", encoding="utf-8") as f:
    f.write(f"{'Index':<6} {'Name':<15} {'Sensors':<8} {'Wells':<8} "
            f"{'Overlap':<8} {'NoSens.':<8} {'Nearby':<8} {'Uncov.':<8} {'Coverage':<8}\n")
    f.write("=" * 80 + "\n")


for i, tensor in enumerate(tqdm(train_tensors_list[:-1], desc="Processing Subjects")):
    
    subject_wells = wells_list[i + 1][subject_name]
    num_sensors = len(subject_wells)

    # ============================================================
    # 1. Tucker Decomposition
    # ============================================================
    tbmd_decomposer = TuckerDecomposer(
        tensors=tensor,
        ranks=None,
        epsilon=1e-2,
        device='mps',
        random_state=SEED
    )

    print(f"\n[Subject {i+1}] Running Tucker Decomposition...")
    tbmd_decomposer.decompose()
    cores = tbmd_decomposer.cores
    factors = tbmd_decomposer.factors

    # ============================================================
    # 2. Modal Tensor Processing (Batch)
    # ============================================================
    madal_processor_config = ModalProcessorConfig(
        device='mps',
        processing_strategy=ProcessingStrategy.BATCH,
        enable_progress_logging=True,
        return_numpy=False
    )

    batch_processor = BatchModalProcessor(madal_processor_config)
    stacker = ModalTensorStacker(madal_processor_config)

    print("[Step] Building modal tensor A...")
    modal_tensors = batch_processor.process_multiple_subjects(cores, factors)
    A_tensor = stacker.stack_modal_tensors(modal_tensors)

    # ============================================================
    # 3. QR-Based Sensor Placement
    # ============================================================
    print("[Step] Performing QR-based sensor placement...")
    qr_decomposer = TensorTubeQRDecomposition(
        tensor=A_tensor,
        N=num_sensors,
        random_state=SEED,
        check_orthogonality=True,
        uniform_distribution=False
    )

    P, Q, R = qr_decomposer.factorize()

    # ============================================================
    # 4. Build Well Matrix & Analyze Overlap
    # ============================================================
    print("[Step] Building wells matrix and analyzing overlap...")
    wells_matrix = build_wells_matrix(wells_list[i + 1], A_tensor.shape, device='mps')
    subject_well_tensor = wells_matrix[subject_name]

    print("-" * 40)
    print(f"[Subject {i+1}] True sensor count: {num_sensors}")
    print("-" * 40)

    # Visualization: Placement
    visualize_combined_placement(P, subject_well_tensor)

    # Full analysis
    analysis_result = analyze_sensor_well_overlap(P, subject_well_tensor, proximity_radius=1.5, proximity_metric='euclidean')
    visualize_overlap_analysis(P, subject_well_tensor, analysis_result)

    save_textual_report(i + 1, subject_name, analysis_result, output_dir="reports")
    append_to_pandas_report(
        subject_index=i + 1,
        subject_name=subject_name,
        analysis_result=analysis_result,
        output_path="reports/summary_all.csv"
    )